# 🚀 VISTA PII Model Trainer (Google Colab T4 GPU)
### Specialized In-Browser Token Classifier for `NAME` & `ADDRESS`

This notebook trains a specialized **MiniLM-L6** model (~23 MB quantized) to detect:
1. **`NAME`** (Indian and global names: lowercase, UPPERCASE, Title Case, honorifics)
2. **`ADDRESS`** (Full Indian addresses: Flat/House, Gali, Sector, Landmark, City, State, PIN)

**Total Training Time**: ~5 minutes on Google Colab free T4 GPU.
**Output**: A featherweight **~18 MB zip** (`pii_browser_model_v2.zip`) that drops directly into `public/models/Xenova/`.

### Step 1: Install Dependencies & Verify GPU
*(Ensure GPU is enabled: `Runtime` > `Change runtime type` > `T4 GPU`)*

In [ ]:
!pip install -q transformers datasets torch onnx onnxruntime faker accelerate onnxscript

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ WARNING: GPU is not active. Go to Runtime > Change runtime type > select T4 GPU.")

### Step 2: Download Fresh Scripts & Generate 100,000 Dataset
*(If `data/train.jsonl` already exists from earlier, it will skip straight to training)*

In [ ]:
import os

# Pull the latest training and evaluation scripts from GitHub
!curl -s -f -O https://raw.githubusercontent.com/helo-ayush/VISTA/main/training/generate_dataset.py
!curl -s -f -O https://raw.githubusercontent.com/helo-ayush/VISTA/main/training/train_pii_model.py
!curl -s -f -O https://raw.githubusercontent.com/helo-ayush/VISTA/main/training/test_inference.py

if not os.path.exists("data/train.jsonl"):
    print("Generating 100,000 authentic Indian & Global records...")
    !python generate_dataset.py
else:
    print("✓ Dataset data/train.jsonl already exists! Ready for training.")

### Step 3: Train MiniLM-L6 with `NAME` & `ADDRESS` Classification
*(Takes ~5 minutes on T4 GPU; automatically exports quantized INT8 ONNX model)*

In [ ]:
# Train for 3 epochs and export directly to onnx_model/
!python train_pii_model.py

### Step 4: Run Inference Benchmark Verification
*(Validates detection on lowercase names, ALL-CAPS names, and addresses in ~4 ms)*

In [ ]:
# Run quick verification across edge cases
!python test_inference.py

### Step 5: Package & Download Ready Browser Model (`pii_browser_model_v2.zip`)
*(Downloads ~18 MB zip file to your local computer)*

In [ ]:
import os
from google.colab import files

zip_filename = "pii_browser_model_v2.zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)

# Create clean zip with all necessary model files
!zip -j {zip_filename} onnx_model/*

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\n✓ Successfully created {zip_filename} ({zip_size_mb:.1f} MB)")
print("Starting browser download...")
files.download(zip_filename)
print("✓ Done! Once downloaded, extract its contents into public/models/Xenova/")